In [1]:
import pandas as pd
import numpy as np
import regex as re
import locationtagger
from deep_translator import GoogleTranslator


In [2]:
product_details = pd.read_csv('product_details.csv')
product_base = pd.read_csv('product_base.csv')

/var/folders/3c/dwwhhxpn5v1_gqzr69zmy0900000gn/T/ipykernel_6748/2588855562.py:1: DtypeWarning: Columns (25,27,28,29,30,31,32,33,34,42,46,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,64,65,66,67,68,69,71,72,74,75,77,79,80,83,84,85,86,87,88,89,90,91,92,93,94,97,99,100,101,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,214,215,216,217,218,219,220,221,222,223,224,225,226,227,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,285,286,287,288,2

In [3]:
product_base.head()

,Unnamed: 0,Referencja,EAN,Nazwa,Dane Produktowe
0,0,97,5010338303006,Blue Dragon - PraÅ¼one algi morskie do sushi n...,https://apps.auchan.pl/feed/getCard/97
1,1,143,5010338303273,Blue Dragon - RyÅ¼ do sushi - 500 g,https://apps.auchan.pl/feed/getCard/143
2,2,1864,5941866617474,Verdino - RoÅlinne plastry pepperoni - 80 g,https://apps.auchan.pl/feed/getCard/1864
3,3,7441,5902506003187,Kolorado - Kostka wc barwiÄ ca do spÅuczki - ...,https://apps.auchan.pl/feed/getCard/7441
4,4,8499,5900143008107,Taverna Dell Ancora - Tortellini z miÄsem - 2...,https://apps.auchan.pl/feed/getCard/8499


In [4]:
product_base = product_base.drop(columns=['Unnamed: 0'])
product_base['EAN'] = product_base['EAN'].apply(str)
product_base['Referencja'] = product_base['Referencja'].apply(str)

In [5]:
product_details.shape

(10261, 654)

In [6]:
product_details.columns.tolist()

['Unnamed: 0',
 'Referencja',
 'waga',
 'zalecenia dla alergikow',
 'opis produktu',
 'marka standaryzowana',
 'informacje dot stylu zycia',
 'jednostka opisowa',
 'kraj pochodzenia',
 'marka',
 'wartość energetyczna',
 'tłuszcz',
 'tym kwasy tłuszczowe nasycone',
 'węglowodany',
 'tym cukry',
 'białko',
 'sól',
 'przygotowanie',
 'błonnik',
 'uzytkowanie i przechowywanie',
 'nazwa produktu ureg prawnie',
 'podmarka',
 'dodatkowe informacje',
 'cechy',
 'dodatki',
 'wartość energetyczna kcal',
 'tłuszcz tym',
 'kwasy nasycone',
 'kwasy jednonienasycone',
 'kwasy wielonienasycone',
 'inne',
 'kwas alfa linolenowy omega',
 'sód',
 'zawartość soli wynika wyłącznie obecności naturalnie występującego sodu',
 'białka',
 'warunki przechowywania',
 'pole rezerwowe 20',
 'pochodzenie',
 'energia',
 'tym kwasy nasycone',
 'witamina',
 'rozszerzona nazwa produktu',
 'rws referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal',
 'Unnamed: 43',
 'kwasy tłuszczowe nasycone',
 'cukry',
 'w

In [7]:
product_details.dropna(axis=1, inplace=True, thresh=5)
product_details.shape

(10261, 263)

In [8]:
product_details.columns.tolist()

['Unnamed: 0',
 'Referencja',
 'waga',
 'zalecenia dla alergikow',
 'opis produktu',
 'marka standaryzowana',
 'informacje dot stylu zycia',
 'jednostka opisowa',
 'kraj pochodzenia',
 'marka',
 'wartość energetyczna',
 'tłuszcz',
 'tym kwasy tłuszczowe nasycone',
 'węglowodany',
 'tym cukry',
 'białko',
 'sól',
 'przygotowanie',
 'błonnik',
 'uzytkowanie i przechowywanie',
 'nazwa produktu ureg prawnie',
 'podmarka',
 'dodatkowe informacje',
 'cechy',
 'dodatki',
 'wartość energetyczna kcal',
 'tłuszcz tym',
 'kwasy nasycone',
 'kwasy jednonienasycone',
 'kwasy wielonienasycone',
 'inne',
 'kwas alfa linolenowy omega',
 'sód',
 'zawartość soli wynika wyłącznie obecności naturalnie występującego sodu',
 'białka',
 'warunki przechowywania',
 'pole rezerwowe 20',
 'pochodzenie',
 'energia',
 'tym kwasy nasycone',
 'witamina',
 'rozszerzona nazwa produktu',
 'rws referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal',
 'Unnamed: 43',
 'kwasy tłuszczowe nasycone',
 'cukry',
 'w

In [9]:
product_details[product_details['tłuszcz surowy'].notna()]

,Unnamed: 0,Referencja,waga,zalecenia dla alergikow,opis produktu,marka standaryzowana,informacje dot stylu zycia,jednostka opisowa,kraj pochodzenia,marka,...,smak kurczaka,jony,tym skrobia,kwas eikozapentaenowy epa,fluorek,cholina,opakowanie zawiera około porcji gramowych,dzienne referencyjne wartości spożycia witamin,około,cynk tlenek cynku
70,70,129967,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów. Zapr...,Purina ONE,NaN,g,NaN,PURINA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
432,432,608509,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów.,Gourmet,NaN,g,NaN,Gourmet,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
521,521,15099,NaN,NaN,Butcher's z kurczakiem to pełnoporcjowa karma...,Butcher's,NaN,kg,Kraj pochodzenia - Wielka Brytania,Butcher's,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590,590,80404,NaN,NaN,Pełnoporcjowa karma dla dorosłych psów.,Friskies,NaN,kg,NaN,riskies,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
636,636,143592,NaN,NaN,Uzupełniająca karma dla dorosłych psów do cod...,Dentalife,NaN,g,NaN,Dentalife,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10123,10123,882978,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów. Karm...,Felix,NaN,g,NaN,Felix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10124,10124,882980,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów. Karm...,Felix,NaN,g,NaN,Felix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10125,10125,882991,NaN,NaN,Pełnoporcjowa karma dla dorosłych kotów. . Ka...,Felix,NaN,g,NaN,Felix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10179,10179,971662,NaN,NaN,Karma uzupełniająca dla dorosłych kotów. biał...,Felix,NaN,g,NaN,Felix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
mapping = {
    "zawartość tłuszczu": "tłuszcz",
    "tłuszcz tym": "tłuszcz",
    "tluszcz": "tłuszcz",
    "tłuszcze": "tłuszcz",
    "tuszcz": "tłuszcz",

    "bialko": "białko",
    "białka": "białko",

    "węglowodany tym": "węglowodany",
    "weglowodany": "węglowodany",

    "tym cukry": "cukry",
    "cukry ogółem": "cukry",
    "cukier": "cukry",

    "energia": "wartość energetyczna",
    "wartość energetyczna kcal": "wartość energetyczna",
    "wartość energetyczna energia kcal": "wartość energetyczna",
    "wartosc energetyczna": "wartość energetyczna",
    "wartosc energetyczna kcal": "wartość energetyczna",
    "blonnik": "wartość energetyczna",
    "wartość energetyczna energia": "wartość energetyczna",
    "energia kcal": "wartość energetyczna",

    "sol": "sól",
}

In [11]:
# Standardizing column names 

for source_col, target_col in mapping.items():
    if source_col in product_details.columns:
        product_details[target_col] = product_details[target_col].combine_first(product_details[source_col])

cols_to_drop = [col for col in product_details.columns if col in mapping]
product_details = product_details.drop(columns=cols_to_drop)

In [12]:
print(product_details.isna().sum().tolist())

[0, 0, 6552, 5951, 4716, 5678, 8959, 5249, 5492, 1593, 4536, 4391, 5563, 4456, 4281, 4595, 9342, 8018, 8822, 2555, 8717, 8845, 7110, 8538, 10185, 10247, 10247, 10227, 10247, 10160, 10225, 8469, 9940, 9245, 9544, 9678, 9435, 10155, 9895, 10133, 4499, 9668, 10244, 10244, 10244, 10246, 10256, 10254, 10251, 10251, 10251, 9984, 10251, 10250, 10255, 10157, 10156, 9994, 10081, 10251, 10037, 10134, 10194, 10206, 10193, 10177, 10178, 10205, 10224, 10241, 10197, 10238, 10238, 10248, 10136, 10250, 10250, 10179, 10208, 10141, 10105, 10182, 10193, 10197, 10239, 10198, 10240, 10131, 10238, 10256, 10237, 10200, 10256, 10253, 10193, 10256, 10252, 10255, 10139, 10174, 10205, 10250, 10254, 10226, 10226, 10256, 10249, 10239, 10238, 10231, 10158, 10159, 10247, 10255, 10214, 10248, 10246, 10171, 10253, 10225, 10246, 10174, 10198, 10253, 10253, 10250, 10249, 10244, 10244, 10253, 10244, 10245, 10255, 10246, 10221, 10201, 10214, 10217, 10207, 10254, 10185, 10244, 10248, 10254, 10246, 10256, 10196, 10253, 1023

In [13]:
product_details.shape

(10261, 243)

In [14]:
cols_to_drop = [col for col in product_details.columns if product_details[col].isna().sum() > 10000]

product_details = product_details.drop(columns=cols_to_drop)

In [15]:
product_details.shape

(10261, 35)

In [16]:
product_details.head()

,Unnamed: 0,Referencja,waga,zalecenia dla alergikow,opis produktu,marka standaryzowana,informacje dot stylu zycia,jednostka opisowa,kraj pochodzenia,marka,...,pole rezerwowe 20,pochodzenie,tym kwasy nasycone,witamina,rozszerzona nazwa produktu,Unnamed: 43,cukry,referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal,wapń,włókno surowe
0,0,97,Waga brutto - 20 g,"ORZECHY - Może zawierać, ORZESZKI ZIEMNE - Mo...",Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,"Produkt odpowiedni dla wegan, Produkt odpowie...",g,Kraj pochodzenia - Korea (Republika Korei),Blue Dragon,...,NaN,NaN,NaN,NaN,NaN,NaN,"6,2 g",NaN,NaN,NaN
1,1,143,Waga brutto - 512 g,"ORZECHY - Może zawierać, ORZESZKI ZIEMNE - Mo...",Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,"Produkt odpowiedni dla wegan, Produkt odpowie...",g,Zapakowano w - Wielka Brytania,Blue Dragon,...,NaN,NaN,NaN,NaN,NaN,NaN,"0,3 g",NaN,NaN,NaN
2,2,1864,80 g,Może zawierać śladowe ilości,Roślinne plastry,NaN,NaN,NaN,Rumunia,Verdino,...,NaN,NaN,NaN,NaN,NaN,NaN,"6,4 g",NaN,NaN,NaN
3,3,7441,NaN,NaN,Blue Ocean. Kostka odświeżająca i czyszcząca ...,NaN,NaN,g,Wyprodukowano w Polsce,Kolorado,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,8499,NaN,"Może zawierać SOJĘ, GORCZYCĘ, SELER, MLEKO, R...",NaN,NaN,NaN,g,Wyprodukowano w Polsce,Taverna Dell'Ancora,...,NaN,NaN,NaN,NaN,NaN,NaN,"0,1 g",NaN,NaN,NaN


In [17]:
print(product_details.isna().sum())

Unnamed: 0                                                              0
Referencja                                                              0
waga                                                                 6552
zalecenia dla alergikow                                              5951
opis produktu                                                        4716
marka standaryzowana                                                 5678
informacje dot stylu zycia                                           8959
jednostka opisowa                                                    5249
kraj pochodzenia                                                     5492
marka                                                                1593
wartość energetyczna                                                 4536
tłuszcz                                                              4391
tym kwasy tłuszczowe nasycone                                        5563
węglowodany                           

In [18]:
cols_to_drop = ['Unnamed: 0',
                'zalecenia dla alergikow',
                'uzytkowanie i przechowywanie',
                'informacje dot stylu zycia', 
                'dodatkowe informacje',
                'dodatki',
                'uzytkowanie i przechowywanie',
                'tym kwasy tłuszczowe nasycone',
                'warunki przechowywania',
                'cechy']
product_details = product_details.drop(columns=cols_to_drop)

In [19]:
product_details['Referencja'] = product_details['Referencja'].apply(str)

In [20]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,...,pole rezerwowe 20,pochodzenie,tym kwasy nasycone,witamina,rozszerzona nazwa produktu,Unnamed: 43,cukry,referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal,wapń,włókno surowe
0,97,Waga brutto - 20 g,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Kraj pochodzenia - Korea (Republika Korei),Blue Dragon,1481 kJ/348 kcal,"<0,1 g",41 g,...,NaN,NaN,NaN,NaN,NaN,NaN,"6,2 g",NaN,NaN,NaN
1,143,Waga brutto - 512 g,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Zapakowano w - Wielka Brytania,Blue Dragon,1469 kJ/346 kcal,"1,0 g","77,2 g",...,NaN,NaN,NaN,NaN,NaN,NaN,"0,3 g",NaN,NaN,NaN
2,1864,80 g,Roślinne plastry,NaN,NaN,Rumunia,Verdino,994 kJ / 240 kcal,"20,4 g","7,1 g",...,NaN,NaN,NaN,NaN,NaN,NaN,"6,4 g",NaN,NaN,NaN
3,7441,NaN,Blue Ocean. Kostka odświeżająca i czyszcząca ...,NaN,g,Wyprodukowano w Polsce,Kolorado,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8499,NaN,NaN,NaN,g,Wyprodukowano w Polsce,Taverna Dell'Ancora,279 kcal,"5,1 g","45,8 g",...,NaN,NaN,NaN,NaN,NaN,NaN,"0,1 g",NaN,NaN,NaN


In [21]:
product_details['waga'] = product_details['waga'].apply(lambda x: re.search(r"\d+\.?\d*", x).group() if isinstance(x, str) and re.search(r"\d+\.?\d*", x) else None)
product_details['waga'] = product_details['waga'].apply(pd.to_numeric)

In [22]:
product_details['kraj pochodzenia'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 10261 entries, 0 to 10260
Series name: kraj pochodzenia
Non-Null Count  Dtype 
--------------  ----- 
4769 non-null   object
dtypes: object(1)
memory usage: 80.3+ KB


In [23]:
# translator = GoogleTranslator(source='auto', target='en')

In [24]:
# product_details['kraj pochodzenia EN'] = product_details['kraj pochodzenia'].apply(lambda x: translator.translate(x) if isinstance(x, str) else None)

In [25]:
# product_details['kraj pochodzenia EN']

In [26]:
# import nltk
# nltk.downloader.download('maxent_ne_chunker')
# nltk.downloader.download('words')
# nltk.downloader.download('treebank')
# nltk.downloader.download('maxent_treebank_pos_tagger')
# nltk.downloader.download('punkt')
# nltk.download('averaged_perceptron_tagger_eng')
# nltk.download('maxent_ne_chunker_tab')

In [27]:
# import geograpy
# import nltk
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('maxent_ne_chunker')
# nltk.download('words')

In [28]:
# product_details['kraj'] = product_details['kraj pochodzenia EN'].apply(lambda x: geograpy.get_geoPlace_context(text = x).countries if isinstance(x, str) else None)

# # product_details['kraj pochodzenia'] = product_details['kraj pochodzenia'].apply(lambda x: x.strip().lower().replace('kraj pochodzenia', '') if isinstance(x, str) else None)
# # product_details['kraj pochodzenia'] = product_details['kraj pochodzenia'].apply(lambda x: x.strip().lower().replace('wyprodukowano w ', '') if isinstance(x, str) else None)

In [29]:
# product_details['kraj'].value_counts()

In [30]:
product_details = product_details.drop(columns=['nazwa produktu ureg prawnie', 'pochodzenie', 'podmarka'])

In [31]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,...,błonnik,pole rezerwowe 20,tym kwasy nasycone,witamina,rozszerzona nazwa produktu,Unnamed: 43,cukry,referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal,wapń,włókno surowe
0,97,20.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Kraj pochodzenia - Korea (Republika Korei),Blue Dragon,1481 kJ/348 kcal,"<0,1 g",41 g,...,NaN,NaN,NaN,NaN,NaN,NaN,"6,2 g",NaN,NaN,NaN
1,143,512.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Zapakowano w - Wielka Brytania,Blue Dragon,1469 kJ/346 kcal,"1,0 g","77,2 g",...,"1,3 g",NaN,NaN,NaN,NaN,NaN,"0,3 g",NaN,NaN,NaN
2,1864,80.0,Roślinne plastry,NaN,NaN,Rumunia,Verdino,994 kJ / 240 kcal,"20,4 g","7,1 g",...,"3,1 g",NaN,NaN,NaN,NaN,NaN,"6,4 g",NaN,NaN,NaN
3,7441,NaN,Blue Ocean. Kostka odświeżająca i czyszcząca ...,NaN,g,Wyprodukowano w Polsce,Kolorado,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8499,NaN,NaN,NaN,g,Wyprodukowano w Polsce,Taverna Dell'Ancora,279 kcal,"5,1 g","45,8 g",...,NaN,NaN,NaN,NaN,NaN,NaN,"0,1 g",NaN,NaN,NaN


In [32]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10261 entries, 0 to 10260
Data columns (total 23 columns):
 #   Column                                                             Non-Null Count  Dtype  
---  ------                                                             --------------  -----  
 0   Referencja                                                         10261 non-null  object 
 1   waga                                                               3676 non-null   float64
 2   opis produktu                                                      5545 non-null   object 
 3   marka standaryzowana                                               4583 non-null   object 
 4   jednostka opisowa                                                  5012 non-null   object 
 5   kraj pochodzenia                                                   4769 non-null   object 
 6   marka                                                              8668 non-null   object 
 7   wartość energetyczna  

In [33]:
cols_to_standardize = ['tłuszcz', 'węglowodany', 'białko', 'sól', 'błonnik', 'cukry']

In [34]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,...,błonnik,pole rezerwowe 20,tym kwasy nasycone,witamina,rozszerzona nazwa produktu,Unnamed: 43,cukry,referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal,wapń,włókno surowe
0,97,20.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Kraj pochodzenia - Korea (Republika Korei),Blue Dragon,1481 kJ/348 kcal,"<0,1 g",41 g,...,NaN,NaN,NaN,NaN,NaN,NaN,"6,2 g",NaN,NaN,NaN
1,143,512.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Zapakowano w - Wielka Brytania,Blue Dragon,1469 kJ/346 kcal,"1,0 g","77,2 g",...,"1,3 g",NaN,NaN,NaN,NaN,NaN,"0,3 g",NaN,NaN,NaN
2,1864,80.0,Roślinne plastry,NaN,NaN,Rumunia,Verdino,994 kJ / 240 kcal,"20,4 g","7,1 g",...,"3,1 g",NaN,NaN,NaN,NaN,NaN,"6,4 g",NaN,NaN,NaN
3,7441,NaN,Blue Ocean. Kostka odświeżająca i czyszcząca ...,NaN,g,Wyprodukowano w Polsce,Kolorado,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8499,NaN,NaN,NaN,g,Wyprodukowano w Polsce,Taverna Dell'Ancora,279 kcal,"5,1 g","45,8 g",...,NaN,NaN,NaN,NaN,NaN,NaN,"0,1 g",NaN,NaN,NaN


In [35]:
for col in cols_to_standardize:
    product_details[col] = product_details[col].apply(lambda x: x.lower().replace("g", "").replace(",", ".").replace("<0.5", "0").replace("<0.1", "0").replace("-", "0").strip() if isinstance(x, str) else None)

In [36]:
for col in cols_to_standardize:
    product_details[col] = (
    product_details[col]
      .astype(str)
      .str.replace(',', '.', regex=False)
      .str.extract(r'([-+]?\d*\.?\d+)')[0]
      .astype(float)
      .round(2)
)

In [37]:
product_details['wartość energetyczna'].to_list()

['1481 kJ/348 kcal',
 '1469 kJ/346 kcal',
 '994 kJ / 240 kcal',
 nan,
 '279 kcal',
 nan,
 nan,
 nan,
 '6 kJ / 1 kcal',
 nan,
 '275/66',
 '2847 kJ/690 kcal',
 nan,
 '163KJ/63 KCAL',
 '155 kJ (37 kcal)',
 nan,
 nan,
 '952 kJ/230 kcal',
 '2403 kJ/ 580 Kcal',
 '2666 kJ / 637 kcal',
 nan,
 nan,
 '591 kJ / 139 kcal',
 nan,
 nan,
 nan,
 nan,
 nan,
 '142 kJ/ 34 Kcal',
 '85 kJ / 20 kcal',
 nan,
 nan,
 '480 kJ/ 115 kcal',
 '945 kJ/ 227 kcal',
 'kcal 254',
 '2360/566',
 '191 kJ/46 kcal',
 nan,
 nan,
 '1630kJ/390 kcal',
 '597 kJ / 143 kcal',
 nan,
 '37 kcal',
 '297 kJ / 71 kcal',
 nan,
 nan,
 '1700 kJ / 400 kcal',
 '1857/444',
 '1627 kJ/ 383 kcal',
 '2399 kJ/578 kcal',
 nan,
 nan,
 '496 kcal',
 '2280 kJ/ 546 Kcal',
 '213 kJ / 51 kcal',
 nan,
 '3389 kJ / 824 kcal',
 '1236 kJ/ 291 Kcal',
 '1248 kj / 295 kcal',
 '294 kJ / 71 kcal',
 nan,
 ' 78 kJ / 18 kcal',
 '2807 kJ / 683 kcal',
 nan,
 nan,
 '1589 kJ / 384 kcal',
 '577 kJ/ 138 Kcal',
 nan,
 nan,
 '445 kcal',
 nan,
 '186 kJ / 44 kcal',
 nan,
 nan,
 

In [38]:
product_details['wartość energetyczna'] = (
    product_details['wartość energetyczna']
        .astype(str)
        .str.split('/')
        .str[-1]
        .str.replace(',', '.', regex=False)
        .str.lower()
        .str.strip()
)

In [39]:
mask = (
    product_details['wartość energetyczna'].str.contains("kj", case=False, na=False)
    & ~product_details['wartość energetyczna'].str.contains("kcal", case=False, na=False)
)


# extract kJ as float
kj = (
    product_details.loc[mask, 'wartość energetyczna']
        .str.extract(r'(\d*\.?\d+)')[0]
        .astype(float)
)

# convert to kcal
kcal = (kj / 4.184).round(2)

# write back (numeric kcal)
product_details.loc[mask, 'wartość energetyczna'] = kcal

In [40]:
product_details.isna().sum()

Referencja                                                              0
waga                                                                 6585
opis produktu                                                        4716
marka standaryzowana                                                 5678
jednostka opisowa                                                    5249
kraj pochodzenia                                                     5492
marka                                                                1593
wartość energetyczna                                                    0
tłuszcz                                                              4413
węglowodany                                                          4468
białko                                                               4295
sól                                                                  4610
przygotowanie                                                        9342
błonnik                               

In [41]:
mask = (
    product_details['wartość energetyczna'].isna()
    & product_details[['tłuszcz', 'węglowodany', 'białko']].notna().all(axis=1)
)

product_details.loc[mask, 'wartość energetyczna'] = (
    product_details.loc[mask, 'tłuszcz'] * 9
    + product_details.loc[mask, 'węglowodany'] * 4
    + product_details.loc[mask, 'białko'] * 4
)

In [42]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,...,błonnik,pole rezerwowe 20,tym kwasy nasycone,witamina,rozszerzona nazwa produktu,Unnamed: 43,cukry,referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal,wapń,włókno surowe
0,97,20.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Kraj pochodzenia - Korea (Republika Korei),Blue Dragon,348 kcal,0.0,41.0,...,NaN,NaN,NaN,NaN,NaN,NaN,6.2,NaN,NaN,NaN
1,143,512.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Zapakowano w - Wielka Brytania,Blue Dragon,346 kcal,1.0,77.2,...,1.3,NaN,NaN,NaN,NaN,NaN,0.3,NaN,NaN,NaN
2,1864,80.0,Roślinne plastry,NaN,NaN,Rumunia,Verdino,240 kcal,20.4,7.1,...,3.1,NaN,NaN,NaN,NaN,NaN,6.4,NaN,NaN,NaN
3,7441,NaN,Blue Ocean. Kostka odświeżająca i czyszcząca ...,NaN,g,Wyprodukowano w Polsce,Kolorado,nan,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8499,NaN,NaN,NaN,g,Wyprodukowano w Polsce,Taverna Dell'Ancora,279 kcal,5.1,45.8,...,NaN,NaN,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN


In [43]:
def extract_kcal(x):
    if isinstance(x, (int, float)) and not pd.isna(x):
        return float(x)
    if isinstance(x, str):
        match = re.search(r'(\d*\.?\d+)\s*kcal', x.lower())
        if match:
            return float(match.group(1))
        return np.nan

    return np.nan


product_details['wartość energetyczna'] = (
    product_details['wartość energetyczna']
        .apply(extract_kcal)
)

In [44]:
product_details['wartość energetyczna'].to_list()[:20]

[348.0,
 346.0,
 240.0,
 nan,
 279.0,
 nan,
 nan,
 nan,
 1.0,
 nan,
 nan,
 690.0,
 nan,
 63.0,
 37.0,
 nan,
 nan,
 230.0,
 580.0,
 637.0]

In [45]:
product_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10261 entries, 0 to 10260
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Referencja       10261 non-null  object
 1   EAN              10261 non-null  object
 2   Nazwa            10261 non-null  object
 3   Dane Produktowe  10261 non-null  object
dtypes: object(4)
memory usage: 320.8+ KB


In [46]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10261 entries, 0 to 10260
Data columns (total 23 columns):
 #   Column                                                             Non-Null Count  Dtype  
---  ------                                                             --------------  -----  
 0   Referencja                                                         10261 non-null  object 
 1   waga                                                               3676 non-null   float64
 2   opis produktu                                                      5545 non-null   object 
 3   marka standaryzowana                                               4583 non-null   object 
 4   jednostka opisowa                                                  5012 non-null   object 
 5   kraj pochodzenia                                                   4769 non-null   object 
 6   marka                                                              8668 non-null   object 
 7   wartość energetyczna  

In [47]:
product_details[product_details['Referencja'] == '43444']

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,...,błonnik,pole rezerwowe 20,tym kwasy nasycone,witamina,rozszerzona nazwa produktu,Unnamed: 43,cukry,referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal,wapń,włókno surowe
1257,43444,NaN,Cebula bogata jest w substancje bakteriobójcz...,NaN,NaN,NaN,Cebula,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
product_details['kategoria'] = np.where(product_details['wartość energetyczna'].notna() | product_base['Nazwa'].str.contains(r'warzywa auchan|owoce auchan|cukiernia auchan|piekarnia auchan|na wagę', case=False, na=False), 'Food', 'Not food')

In [49]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10261 entries, 0 to 10260
Data columns (total 24 columns):
 #   Column                                                             Non-Null Count  Dtype  
---  ------                                                             --------------  -----  
 0   Referencja                                                         10261 non-null  object 
 1   waga                                                               3676 non-null   float64
 2   opis produktu                                                      5545 non-null   object 
 3   marka standaryzowana                                               4583 non-null   object 
 4   jednostka opisowa                                                  5012 non-null   object 
 5   kraj pochodzenia                                                   4769 non-null   object 
 6   marka                                                              8668 non-null   object 
 7   wartość energetyczna  

In [50]:
product_details.head()

,Referencja,waga,opis produktu,marka standaryzowana,jednostka opisowa,kraj pochodzenia,marka,wartość energetyczna,tłuszcz,węglowodany,...,pole rezerwowe 20,tym kwasy nasycone,witamina,rozszerzona nazwa produktu,Unnamed: 43,cukry,referencyjna wartość spożycia dla przeciętnej osoby dorosłej kcal,wapń,włókno surowe,kategoria
0,97,20.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Kraj pochodzenia - Korea (Republika Korei),Blue Dragon,348.0,0.0,41.0,...,NaN,NaN,NaN,NaN,NaN,6.2,NaN,NaN,NaN,Food
1,143,512.0,Odpowiedni dla Wegetarian i Wegan.,Blue Dragon,g,Zapakowano w - Wielka Brytania,Blue Dragon,346.0,1.0,77.2,...,NaN,NaN,NaN,NaN,NaN,0.3,NaN,NaN,NaN,Food
2,1864,80.0,Roślinne plastry,NaN,NaN,Rumunia,Verdino,240.0,20.4,7.1,...,NaN,NaN,NaN,NaN,NaN,6.4,NaN,NaN,NaN,Food
3,7441,NaN,Blue Ocean. Kostka odświeżająca i czyszcząca ...,NaN,g,Wyprodukowano w Polsce,Kolorado,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not food
4,8499,NaN,NaN,NaN,g,Wyprodukowano w Polsce,Taverna Dell'Ancora,279.0,5.1,45.8,...,NaN,NaN,NaN,NaN,NaN,0.1,NaN,NaN,NaN,Food


In [51]:
product_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10261 entries, 0 to 10260
Data columns (total 24 columns):
 #   Column                                                             Non-Null Count  Dtype  
---  ------                                                             --------------  -----  
 0   Referencja                                                         10261 non-null  object 
 1   waga                                                               3676 non-null   float64
 2   opis produktu                                                      5545 non-null   object 
 3   marka standaryzowana                                               4583 non-null   object 
 4   jednostka opisowa                                                  5012 non-null   object 
 5   kraj pochodzenia                                                   4769 non-null   object 
 6   marka                                                              8668 non-null   object 
 7   wartość energetyczna  

In [52]:
from unidecode import unidecode

product_base['Nazwa'] = product_base['Nazwa'].apply(lambda x: unidecode(x))

In [53]:
product_details = product_details.fillna("")

In [54]:
product_details.to_csv('product_details_cleaned.csv')
product_base.to_csv('product_base_cleaned.csv')